In [ ]:
!python train.py

In [ ]:
import torch
import torch.nn.functional as F
from data import get_dataloaders
from sts_transformer import STSTransformer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

_, vocab = get_dataloaders(
    train_path='train.csv',
    val_path='val.csv',
    batch_size=32,
    max_len=128,
)
print(f'Vocab size: {len(vocab):,}')

model = STSTransformer(
    vocab_size=len(vocab),
    d_model=256,
    n_layers=4,
    n_heads=8,
    d_ff=512,
    dropout=0.2,
    max_len=128,
).to(DEVICE)

model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
model.eval()
print('Model loaded successfully!')

In [ ]:
def similarity(sent1: str, sent2: str) -> float:
    ids_a = torch.tensor([vocab.encode(sent1)], dtype=torch.long).to(DEVICE)
    ids_b = torch.tensor([vocab.encode(sent2)], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        score = model(ids_a, ids_b)
    return score.item()

def show_similarity(sent1: str, sent2: str):
    score = similarity(sent1, sent2)

    print(f'Sentence 1 : {sent1}')
    print(f'Sentence 2 : {sent2}')
    print(f'Score      : {score:.4f}')
    print('-' * 60)

print(' Functions ready!')

In [ ]:
test_pairs = [
    ('A dog is running in the park',        'A dog is playing outside'),
    ('The man is eating a sandwich',        'A person is having lunch'),
    ('A woman is singing a song',           'A lady is performing music'),


    ('The cat sat on the mat',              'A dog is running in the park'),
    ('A man is riding a bicycle',           'Someone is driving a car'),


    ('I love pizza',                        'The stock market crashed today'),
    ('The sky is blue',                     'She is cooking dinner'),
]

print('=' * 60)
for s1, s2 in test_pairs:
    show_similarity(s1, s2)

In [ ]:
sentence1 = 'Islamist revival movements gained followers across the Muslim world, but failed to secure political power except in Iran and Sudan'
sentence2 = 'The majority of Muslims want the Islam revival to happen'

show_similarity(sentence1, sentence2)

In [ ]:
pairs = [
    ('A dog is running in the park',   'A dog is playing outside'),
    ('The man is eating a sandwich',   'A person is having lunch'),
    ('The cat sat on the mat',         'A dog is running in the park'),
    ('I love pizza',                   'The stock market crashed today'),
]

print(f'{"Sentence 1":<40} {"Sentence 2":<40} {"Score":>6}')
print('=' * 92)
for s1, s2 in pairs:
    score = similarity(s1, s2)
    bar = '█' * int(score * 20)
    print(f'{s1:<40} {s2:<40} {score:>6.3f}  {bar}')